In [ ]:
# Base path to your volume
VOLUME_PATH = "/Volumes/workspace/default/trade-lake"

# Read alerts
spark.sql("DROP TABLE IF EXISTS workspace.bronze.alerts")
alerts_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{VOLUME_PATH}/alerts.csv")
)

print(f"Alerts: {alerts_df.count()} rows")
alerts_df.printSchema()
alerts_df.show(5)

In [ ]:
orders_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{VOLUME_PATH}/orders.csv")
)

trades_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{VOLUME_PATH}/trades.csv")
)

print(f"Orders: {orders_df.count()} rows")
print(f"Trades: {trades_df.count()} rows")

In [ ]:
# First, create a dedicated bronze schema to keep things organized
spark.sql("DROP TABLE IF EXISTS workspace.bronze.alerts")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")

# Write each DataFrame as a Delta table
alerts_df.write.mode("overwrite").saveAsTable("workspace.bronze.alerts")
orders_df.write.mode("overwrite").saveAsTable("workspace.bronze.orders")
trades_df.write.mode("overwrite").saveAsTable("workspace.bronze.trades")

print("Bronze tables created successfully")

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

# Create silver schema
spark.sql("DROP TABLE IF EXISTS workspace.silver.alerts")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

# Read from bronze
alerts_bronze = spark.table("workspace.bronze.alerts")

# Apply silver transformations
alerts_silver = (
    alerts_bronze
    .withColumn("match_count", F.col("match_count").cast(IntegerType()))
    .withColumn("alert_type", F.lower(F.col("alert_type")))
    .withColumn("symbol", F.upper(F.col("symbol")))
    .dropDuplicates(["alert_type", "trader_id", "symbol", "window_start", "window_end"])
)

# Write to silver
alerts_silver.write.mode("overwrite").saveAsTable("workspace.silver.alerts")

print(f"Silver alerts: {alerts_silver.count()} rows")
alerts_silver.printSchema()

In [ ]:
# Orders silver
orders_bronze = spark.table("workspace.bronze.orders")

orders_silver = (
    orders_bronze
    .withColumn("side", F.lower(F.col("side")))
    .withColumn("status", F.lower(F.col("status")))
    .withColumn("symbol", F.upper(F.col("symbol")))
    .withColumn("trader_type", F.lower(F.col("trader_type")))
    .dropDuplicates(["order_id", "status"])
)

orders_silver.write.mode("overwrite").saveAsTable("workspace.silver.orders")
print(f"Silver orders: {orders_silver.count()} rows")

# Trades silver
trades_bronze = spark.table("workspace.bronze.trades")

trades_silver = (
    trades_bronze
    .withColumn("side", F.lower(F.col("side")))
    .withColumn("symbol", F.upper(F.col("symbol")))
    .withColumn("trader_type", F.lower(F.col("trader_type")))
    .dropDuplicates(["trade_id"])
)

trades_silver.write.mode("overwrite").saveAsTable("workspace.silver.trades")
print(f"Silver trades: {trades_silver.count()} rows")

In [ ]:
# Gold table 1: trader risk summary with UNION for wash trading pairs
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

trader_risk = spark.sql("""
    WITH all_flagged_traders AS (
        -- Spoofing alerts use trader_id
        SELECT
            trader_id,
            symbol,
            alert_type,
            cancel_rate,
            cancel_count,
            detected_at
        FROM workspace.silver.alerts
        WHERE trader_id IS NOT NULL

        UNION ALL

        -- Wash trading alerts use trader_id_1
        SELECT
            trader_id_1 AS trader_id,
            symbol,
            alert_type,
            cancel_rate,
            cancel_count,
            detected_at
        FROM workspace.silver.alerts
        WHERE trader_id_1 IS NOT NULL

        UNION ALL

        -- Wash trading alerts also use trader_id_2 (the partner)
        SELECT
            trader_id_2 AS trader_id,
            symbol,
            alert_type,
            cancel_rate,
            cancel_count,
            detected_at
        FROM workspace.silver.alerts
        WHERE trader_id_2 IS NOT NULL
    )
    SELECT
        trader_id,
        COUNT(*) AS total_alerts,
        COUNT(DISTINCT symbol) AS distinct_symbols_flagged,
        COUNT(DISTINCT alert_type) AS distinct_alert_types,
        AVG(cancel_rate) AS avg_cancel_rate,
        MAX(cancel_count) AS max_cancel_count_in_window,
        MIN(detected_at) AS first_alert_at,
        MAX(detected_at) AS latest_alert_at
    FROM all_flagged_traders
    GROUP BY trader_id
    ORDER BY total_alerts DESC
""")

trader_risk.write.mode("overwrite").saveAsTable("workspace.gold.trader_risk_summary")
trader_risk.show(10, truncate=False)

In [ ]:
spark.sql("""
    SELECT
        alert_type,
        COUNT(*) AS total,
        COUNT(trader_id) AS has_trader_id,
        COUNT(trader_id_1) AS has_trader_id_1,
        COUNT(trader_id_2) AS has_trader_id_2
    FROM workspace.silver.alerts
    GROUP BY alert_type
""").show()

In [ ]:
# Gold table 2: asset alert activity
asset_activity = spark.sql("""
    SELECT
        symbol,
        alert_type,
        COUNT(*) AS alert_count,
        AVG(cancel_rate) AS avg_cancel_rate,
        MIN(detected_at) AS first_detected,
        MAX(detected_at) AS latest_detected
    FROM workspace.silver.alerts
    GROUP BY symbol, alert_type
    ORDER BY alert_count DESC
""")

asset_activity.write.mode("overwrite").saveAsTable("workspace.gold.asset_alert_activity")
asset_activity.show(truncate=False)

In [ ]:
# Gold table 3: daily alert metrics
daily_metrics = spark.sql("""
    SELECT
        DATE(detected_at) AS alert_date,
        alert_type,
        COUNT(*) AS total_alerts,
        COUNT(DISTINCT COALESCE(trader_id, trader_id_1)) AS unique_traders,
        COUNT(DISTINCT symbol) AS unique_symbols,
        AVG(cancel_rate) AS avg_cancel_rate
    FROM workspace.silver.alerts
    GROUP BY DATE(detected_at), alert_type
    ORDER BY alert_date DESC, alert_type
""")

daily_metrics.write.mode("overwrite").saveAsTable("workspace.gold.daily_alert_metrics")
daily_metrics.show(truncate=False)